In [ ]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/products_asos.csv")

# Supprimer les lignes avec des valeurs manquantes
df = df.dropna()


In [2]:
#counting words that comes the most in a column for creating a new column/feature
from collections import Counter

all_words = " ".join(df["name"].str.lower()).split()
Counter(all_words).most_common(10)

[('in', 30895),
 ('dress', 11000),
 ('asos', 8330),
 ('black', 7203),
 ('design', 7042),
 ('with', 6023),
 ('mini', 4376),
 ('sleeve', 4100),
 ('midi', 3659),
 ('top', 3631)]

In [ ]:
# ── 2. Brand (liste blanche + fallback regex mots capitalisés) ────────────────
# Triée du plus long au plus court pour éviter qu'ASOS matche avant ASOS DESIGN
KNOWN_BRANDS = sorted([
    'ASOS DESIGN', 'ASOS EDITION', 'ASOS LUXE', 'ASOS 4505', 'ASOS Weekend Collective', 'ASOS',
    'The North Face', 'Under Armour', 'New Balance', 'New Look',
    'adidas', 'Nike', 'Puma', 'Reebok', 'ellesse', 'Fila', 'Vans', 'Converse',
    'Tommy Hilfiger', 'Tommy Jeans', 'Calvin Klein', 'Polo Ralph Lauren',
    'Lacoste', 'Fred Perry', 'HUGO', 'Napapijri', 'Barbour', 'Columbia',
    'Jack Wolfskin', 'Carhartt WIP', 'Carhartt', 'Berghaus', 'Dickies',
    'River Island', 'Miss Selfridge', 'Noisy May', 'Vero Moda', 'Mango',
    'Stradivarius', 'Bershka', 'Pull&Bear', 'Monki', 'Weekday', 'Lindex',
    'AllSaints', 'French Connection', 'Daisy Street', 'Pretty Lavish',
    'Little Mistress', 'Jaded Rose', 'Rare London', 'Club L London',
    'In The Style', 'I Saw It First', 'Reclaimed Vintage', 'Native Youth',
    'Sister Jane', 'Something New', 'Flounce London', 'Closet London',
    'Love Triangle', 'True Violet', 'South Beach', 'Urban Bliss', 'Pink Soda',
    'Free People', 'Gym King', 'Sixth June', 'Ann Summers', 'Bluebella', 'Dr Denim',
    "Levi's", 'COLLUSION', 'ASYOU', 'HIIT', 'TALA', 'LAPP', 'TFNC', 'JJXX', 'JDY',
    'DKNY', 'VAI21', 'SNDYS', 'TSTM',
], key=len, reverse=True)
 
def extract_brand(name):
    name = str(name)
    # 1. Chercher dans la liste blanche (insensible à la casse)
    for brand in KNOWN_BRANDS:
        if name.lower().startswith(brand.lower()):
            return brand
    # 2. Fallback : mots capitalisés au début du nom
    match = re.match(r'^([A-Z][A-Za-z0-9&\-\'\.]*(?:\s+[A-Z][A-Za-z0-9&\-\'\.]*)*)', name)
    return match.group(1).strip() if match else name.split()[0]
 
df['brand'] = df['name'].apply(extract_brand)

# ─────────────────────────────────────────────
# 3. CATÉGORIE SIMPLIFIÉE (depuis le nom du produit)
# ─────────────────────────────────────────────


def categorize(nom):
    nom = str(nom).lower()

    if ('coat' in nom or 'jacket' in nom or 'puffer' in nom or 'bomber' in nom or 'biker' in nom or
        'gilet' in nom or 'parka' in nom or 'trench' in nom or 'shacket' in nom or 'windbreaker' in nom):
        return 'outerwear'

    elif ('dress' in nom or 'jumpsuit' in nom or 'playsuit' in nom):
        return 'dress'

    elif 'skirt' in nom:
        return 'skirt'

    elif ('jeans' in nom or 'trouser' in nom or 'shorts' in nom or 'legging' in nom or
          'jogger' in nom or 'pant' in nom or 'denim' in nom):
        return 'bottom'

    elif ('shirt' in nom or ' top' in nom or 'blouse' in nom or 'vest' in nom or 'hoodie' in nom or
          'sweatshirt' in nom or 'jumper' in nom or 'cardigan' in nom or 'tee' in nom or
          'knitwear' in nom or 'crop' in nom or 'knit' in nom or 'sweater' in nom):
        return 'top'

    else:
        return 'other'

df['category'] = df['name'].apply(categorize)


# ─────────────────────────────────────────────
# 4. COULEUR (nettoyage + base_color)
# ─────────────────────────────────────────────

# Mettre en minuscules
df['color'] = df['color'].str.lower()

# Mapper vers une couleur de base (10 couleurs)
def get_base_color(color):
    c = str(color).lower()

    if ('grey' in c or 'gray' in c or 'silver' in c or 'charcoal' in c or 'slate' in c or 'ash' in c):
        return 'grey'

    elif ('blue' in c or 'navy' in c or 'cobalt' in c or 'teal' in c or 'denim' in c or
          'aqua' in c or 'turquoise' in c or 'indigo' in c):
        return 'blue'

    elif ('green' in c or 'sage' in c or 'olive' in c or 'mint' in c or 'forest' in c or
          'emerald' in c or 'jade' in c or 'khaki' in c):
        return 'green'

    elif ('purple' in c or 'lilac' in c or 'lavender' in c or 'violet' in c or
          'mauve' in c or 'plum' in c or 'aubergine' in c):
        return 'purple'

    elif ('red' in c or 'scarlet' in c or 'crimson' in c or 'cherry' in c or
          'wine' in c or 'maroon' in c or 'burgundy' in c or 'berry' in c):
        return 'red'

    elif ('orange' in c or 'amber' in c or 'rust' in c or 'terracotta' in c):
        return 'orange'

    elif ('brown' in c or 'tan' in c or 'caramel' in c or 'coffee' in c or
          'chocolate' in c or 'mocha' in c):
        return 'brown'

    elif ('pink' in c or 'rose' in c or 'blush' in c or 'coral' in c or
          'salmon' in c or 'fuchsia' in c):
        return 'pink'

    elif ('white' in c or 'ivory' in c or 'cream' in c or 'off-white' in c or 'snow' in c):
        return 'white'
    
    elif ('black' in c or 'ebony' in c or 'onyx' in c):
        return 'black'
    
    elif ('beige' in c or 'nude' in c or 'sand' in c or 'wheat' in c or 'ecru' in c):
        return 'beige'
    
    else:
        return 'other'
    
df['base_color'] = df['color'].apply(get_base_color)


# ─────────────────────────────────────────────
# 5. TAILLES (parsing de la colonne "size")
# ─────────────────────────────────────────────
# Exemple : "UK 4,UK 6,UK 8 - Out of stock,UK 10"
# → total=4, available=3, oos=1

def parser_tailles(size_str):
    if pd.isna(size_str):
        return 0, 0, 0, False, False
    tailles = [s.strip() for s in str(size_str).split(',')]
    total        = len(tailles)
    nb_oos       = sum(1 for s in tailles if 'Out of stock' in s)
    nb_dispo     = total - nb_oos
    has_oos      = nb_oos > 0
    fully_oos    = nb_dispo == 0
    return total, nb_dispo, nb_oos, has_oos, fully_oos

df[['nb_sizes_total',
    'nb_sizes_available',
    'nb_sizes_out_of_stock',
    'has_out_of_stock',
    'fully_out_of_stock']] = df['size'].apply(lambda x: pd.Series(parser_tailles(x)))


# ─────────────────────────────────────────────
# 6. SEGMENT DE PRIX
# ─────────────────────────────────────────────

def get_price_segment(price):
    if price < 20:
        return '<£20'
    elif price < 50:
        return '£20-50'
    elif price < 100:
        return '£50-100'
    else:
        return '>£100'

df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['price_segment'] = df['price'].apply(get_price_segment)


# ─────────────────────────────────────────────
# 7. BRAND GROUP  ← colonne bonus demandée
# ─────────────────────────────────────────────
# Regroupe les variantes d'une même marque :
# "ASOS DESIGN Tall", "ASOS DESIGN Curve" → "ASOS DESIGN"
# "Nike Running", "Nike Training"          → "Nike"
# "The North Face Running"                 → "The North Face"
# etc.

# On liste les "BRANDSs" dans l'ordre du plus long au plus court,
# pour éviter qu'un préfixe court masque un préfixe plus spécifique.
BRANDS = [
    'ASOS DESIGN','ASOS EDITION','ASOS LUXE',
    'ASOS Weekend Collective','ASOS 4505','ASOS',
    'The North Face','Under Armour','New Balance','adidas','Nike',
    'Puma','Reebok','Jack Wolfskin','Napapijri','Berghaus',
    'Columbia','Dickies','Vans','Converse','Fila',
    'New Look','Noisy May','River Island','Miss Selfridge',
    'Vero Moda','Mango','Stradivarius','Bershka','Pull&Bear',
    'Tommy Hilfiger','Tommy Jeans','Calvin Klein',
    'Polo Ralph Lauren','AllSaints','Fred Perry','Lacoste','HUGO','Barbour',
    'Daisy Street','French Connection','Pretty Lavish','Little Mistress',
    'Jaded Rose','Rare London','Club L London','In The Style',
    'I Saw It First','Reclaimed Vintage','Native Youth','Sister Jane',
    'Dream Sister Jane','Something New','Flounce London','Closet London',
    'Twisted Wunder','Love Triangle','True Violet','Fashion Union',
    'South Beach','Urban Bliss','Pink Soda','Free People',
    'Gym King','Sixth June','Ann Summers','Bluebella','Dr Denim',
    'COLLUSION','ASYOU','HIIT','TALA','LAPP','TFNC',
    'Lindex','Monki','Weekday',
    'Carhartt WIP','Carhartt','Allsaints','PUMA'
]

def get_brand_group(brand):
    brand_str = str(brand)
    for br in BRANDS:
        if brand_str.startswith(br):       # ← br = la marque courante
            if br in ('Allsaints',):
                return 'AllSaints'
            if br in ('PUMA',):
                return 'Puma'
            if br in ('Dream Sister Jane',):
                return 'Sister Jane'
            return br
    # Fallback : premier mot de la marque
    return brand_str.split()[0] if brand_str else 'Unknown'

df['brand_group'] = df['brand'].apply(get_brand_group)


# ─────────────────────────────────────────────
# 8. SÉLECTION & EXPORT DES COLONNES FINALES
# ─────────────────────────────────────────────

colonnes_finales = [
    'category',
    'price',
    'color',
    'brand',
    'brand_group',     
    'base_color',
    'has_out_of_stock',
    'nb_sizes_total',
    'nb_sizes_available',
    'nb_sizes_out_of_stock',
    'fully_out_of_stock',
    'price_segment',
]

df_clean = df[colonnes_finales]

df_clean.to_csv("../data/processed/asos_clean.csv", index=False)